# Capstone 4 Variant Comparison Notebook
Notebook equivalent of the Streamlit compare app.

This notebook runs the same four variants:
- base
- base_plus_rag
- fine_tuned
- fine_tuned_plus_rag

and computes the same metrics and charts.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import HTML, display

from src.config.settings import load_settings
from src.eval.metrics import approx_token_count, correctness_proxy, format_adherence_score
from src.inference.adapter_registry import resolve_latest_adapter
from src.inference.router import InferenceRouter

In [ ]:
# Ensure current working directory is the project root if possible.
cwd = Path.cwd().resolve()
project_root = cwd
if not (project_root / 'src').exists():
    for p in [cwd, *cwd.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            project_root = p
            break

if Path.cwd().resolve() != project_root:
    import os
    os.chdir(project_root)

print('Working dir:', Path.cwd().resolve())

In [ ]:
def _format_expected(record: dict) -> str:
    out = record.get('output', {})
    fields = [
        'Severity',
        'FileLine',
        'Category',
        'Issue',
        'WhyItMatters',
        'SuggestedFix',
        'PatchSnippet',
        'Confidence',
    ]
    return '\n'.join([f"{f}: {out.get(f, '')}" for f in fields])


def _build_prompt(record: dict) -> str:
    instruction = str(record.get('task_instruction', '')).strip()
    inp = record.get('input', {})
    return (
        f"{instruction}\n\n"
        f"File: {inp.get('file_path', 'unknown')}:{inp.get('line_hint', '')}\n"
        f"Context: {inp.get('context_summary', '')}\n\n"
        f"Changed code:\n{inp.get('changed_code', '')}\n"
    )


def _metric_row(result: dict, expected: dict | None = None) -> dict:
    text = result.get('text', '')
    row = {
        'variant': result.get('variant'),
        'format_adherence': format_adherence_score(text),
        'latency_ms': float(result.get('latency_ms', 0.0)),
        'token_estimate': approx_token_count(text),
    }
    if expected is not None:
        proxy = correctness_proxy(text, expected)
        row.update(proxy)
    return row


def _escape_html(text: str) -> str:
    return (text.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;'))


def _render_variant_cards(results: list[dict], output_box_height: int = 420) -> None:
    cards = []
    for result in results:
        variant = result.get('variant', '')
        text = _escape_html(result.get('text', ''))
        latency = result.get('latency_ms', 0.0)
        meta = result.get('metadata') or {}
        meta_str = ' | '.join([f"{k}: {v}" for k, v in meta.items()])
        meta_html = f"latency: {latency} ms" + (f" &nbsp;|&nbsp; { _escape_html(meta_str) }" if meta_str else '')

        cards.append(
            f'''
            <div class="variant-item">
              <div class="variant-header">{_escape_html(variant)}</div>
              <div class="variant-meta">{meta_html}</div>
              <div class="variant-card">{text}</div>
            </div>
            '''
        )

    html = f'''
    <style>
      .variant-grid {{
        display: grid;
        grid-template-columns: 1fr 1fr;
        gap: 12px;
      }}
      .variant-header {{
        font-size: 15px;
        font-weight: 700;
        margin-bottom: 4px;
      }}
      .variant-meta {{
        font-size: 12px;
        color: #666;
        margin-bottom: 6px;
      }}
      .variant-card {{
        border: 1px solid #d0d0d0;
        border-radius: 8px;
        padding: 12px;
        height: {output_box_height}px;
        overflow-y: auto;
        background: #fafafa;
        white-space: pre-wrap;
        font-family: ui-monospace, Menlo, Consolas, monospace;
        font-size: 12px;
      }}
    </style>
    <div class="variant-grid">
      {''.join(cards)}
    </div>
    '''
    display(HTML(html))

In [ ]:
settings = load_settings()
latest_adapter = resolve_latest_adapter(settings.models_dir) or settings.adapter_path
print('Adapter path in use:', latest_adapter)

eval_rows: list[dict] = []
if settings.eval_file.exists():
    with settings.eval_file.open('r', encoding='utf-8') as f:
        eval_rows = [json.loads(line) for line in f if line.strip()]

print('Eval rows loaded:', len(eval_rows))
router = InferenceRouter(adapter_path=latest_adapter)

## Configure Input
Set prompt_mode to custom or eval and then run the next cell.

In [ ]:
prompt_mode = 'custom'  # 'custom' or 'eval'
eval_row_index = 0
custom_prompt = (
    'Review this code change and return structured review comments with '
    'Severity, Category, Issue, WhyItMatters, SuggestedFix, and Confidence.'
)

if prompt_mode == 'eval':
    assert eval_rows, 'Eval dataset is empty. Switch to custom mode or load eval rows.'
    assert 0 <= eval_row_index < len(eval_rows), 'eval_row_index is out of range.'
    selected_row = eval_rows[eval_row_index]
    prompt_text = _build_prompt(selected_row)
    print('Using eval row:', eval_row_index)
    print('Prompt preview:
')
    print(prompt_text[:1000])
    print('
Expected output (ground truth):
')
    print(_format_expected(selected_row))
else:
    selected_row = None
    prompt_text = custom_prompt
    print('Using custom prompt:
')
    print(prompt_text)

## Run Comparison
This executes all four variants and renders fixed-size result cards.

In [ ]:
results = router.run_all_with_timings(prompt_text)
metrics = [_metric_row(r, selected_row) for r in results]
metric_df = pd.DataFrame(metrics)

print('Variant Outputs')
_render_variant_cards(results, output_box_height=420)

display(metric_df)

## Charts
Equivalent charts to the Streamlit app.

In [ ]:
fig1 = px.bar(metric_df, x='variant', y='format_adherence', title='Format Adherence by Variant', range_y=[0, 1])
fig2 = px.bar(metric_df, x='variant', y='latency_ms', title='Latency by Variant (ms)')
fig3 = px.bar(metric_df, x='variant', y='token_estimate', title='Token Estimate by Variant')

fig1.show()
fig2.show()
fig3.show()

if 'category_match' in metric_df.columns and 'severity_match' in metric_df.columns:
    fig4 = px.bar(
        metric_df,
        x='variant',
        y=['category_match', 'severity_match', 'lexical_overlap'],
        barmode='group',
        title='Correctness Proxy Metrics',
    )
    fig4.show()